# Where did we go wrong?? 
## Breaking down our hosting capacity calculations

In [1]:
import os
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import box, MultiLineString

In [2]:
os.environ['PROJ_LIB'] = '/opt/anaconda3/share/proj'

In [11]:
# set option to see all data frame columns
pd.set_option('display.max_columns', None)

# Investigating FeederName: AIRWAYS 1103

In [72]:
# Read in circuits, and crop to a single feederline
pge_circuits = gpd.read_file("../../../../../capstone/electrigrid/data/utilities/pge_shapefiles/LineDetail/LineDetail.shp").to_crs("EPSG:4326")

airway = pge_circuits[pge_circuits['FeederName'] == "AIRWAYS 1103"]

### Convert all MW columns to KW

In [5]:
vars = ['GenCapacit', 'GenericPVC', 'GenCapac_1', 'GenericCap', 'LoadCapaci']

for col in vars:
    pge_circuits[col] = pge_circuits[col] / 1000

In [12]:
pge_circuits.head(2)

,FeederId,FeederName,globalid,CSV_LineSe,LoadCapaci,voltage_kv,phase_cnt,limiting_m,limiting_c,ICA_Analys,lica_analy,Division,GenCapacit,GenericPVC,GenCapac_1,GenericCap,limiting_1,limiting_2,limiting_3,limiting_4,limiting_5,limiting_6,limiting_7,limiting_8,ScreenL,Publish,Last_Updat,SHAPE_Leng,geometry
4320,252041103,AIRWAYS 1103,{A940F439-D002-4B25-B2FA-C605AC57D5EC},3456531,0.0,12.0,3,None,None,Apr 2025,Apr 2025,Fresno,0.0,0.0,0.0,0.0,None,None,None,None,None,None,None,None,Likely to pass,1,2025-06-02,4.576772,"LINESTRING (-119.68330 36.77218, -119.68324 36..."
7817,252041103,AIRWAYS 1103,{3DADAF5B-DE62-44A6-AC95-38028C09E0B0},4777776,0.0,12.0,3,None,None,Apr 2025,Apr 2025,Fresno,0.0,0.0,0.0,0.0,None,None,None,None,None,None,None,None,Likely to pass,1,2025-06-02,2.777985,"LINESTRING (-119.68609 36.78440, -119.68609 36..."


### Remove unecessary rows

In [14]:
airway = airway[['FeederId', 'LoadCapaci', 'GenCapacit', 'GenericPVC', 'GenCapac_1', 'GenericCap', 'geometry']]
airway.head(2)

,FeederId,LoadCapaci,GenCapacit,GenericPVC,GenCapac_1,GenericCap,geometry
4320,252041103,0.0,0.0,0.0,0.0,0.0,"LINESTRING (-119.68330 36.77218, -119.68324 36..."
7817,252041103,0.0,0.0,0.0,0.0,0.0,"LINESTRING (-119.68609 36.78440, -119.68609 36..."


In [16]:
airway['GenCapacit'].describe()

count    282.0
mean       0.0
std        0.0
min        0.0
25%        0.0
50%        0.0
75%        0.0
max        0.0
Name: GenCapacit, dtype: float64

In [17]:
airway['LoadCapaci'].describe()

count    282.0
mean       0.0
std        0.0
min        0.0
25%        0.0
50%        0.0
75%        0.0
max        0.0
Name: LoadCapaci, dtype: float64

## Conclusion: the selected feederline didn't have any data, explaining why we had null results :)

# Investigating feederlines where values = 0

In [28]:
nrow = pge_circuits.count().iloc[1]
nrow_feeders = pge_circuits.groupby('FeederId').count().shape[0]

731431

In [61]:
nrow_feeders

2928

### Generation capacity

In [30]:
# how many observations have generation capacity = to 0?
gencap_0 = pge_circuits[pge_circuits['GenCapacit'] == 0].shape[0]

print(f"{gencap_0}, or {round(gencap_0/nrow * 100)}% of observations contain generation capacity = 0")

574237, or 79% of observations contain generation capacity = 0


In [65]:
# how many feederlines is that?
gencap_0_feeders = pge_circuits[pge_circuits['GenCapacit'] == 0].groupby('FeederId').count().shape[0]
df = pge_circuits.groupby('FeederId').sum('GenCapacit')
gencap_0_feeders_complete = df[df['GenCapacit'] == 0].count().iloc[0]

print(f"{gencap_0_feeders}, or {round(gencap_0_feeders/nrow_feeders * 100)}% of feederlines contain generation capacity = 0")
print(f"{gencap_0_feeders_complete}, or {round(gencap_0_feeders_complete/nrow_feeders * 100)} % of ENTIRE feederlines have 0 generation capacity.")

2320, or 79% of feederlines contain generation capacity = 0
1641, or 56 % of ENTIRE feederlines have 0 generation capacity.


### Generation capacity PV

In [49]:
# how many observations have generation capacity = to 0?
pv_0 = pge_circuits[pge_circuits['GenericPVC'] == 0].shape[0]

print(f"{pv_0}, or {round(pv_0/nrow * 100)}% of observations contain PV generation capacity = 0")

572625, or 78% of observations contain generation capacity = 0


In [63]:
# how many feederlines is that?
pv_0_feeders = pge_circuits[pge_circuits['GenericPVC'] == 0].groupby('FeederId').count().shape[0]
df = pge_circuits.groupby('FeederId').sum('GenericPVC')
pv_0_feeders_complete = df[df['GenericPVC'] == 0].count().iloc[0]

print(f"{pv_0_feeders}, or {round(pv_0_feeders/nrow_feeders * 100)}% of feederlines contain PV generation capacity = 0")
print(f"{pv_0_feeders_complete}, or {round(pv_0_feeders_complete/nrow_feeders * 100)} % of ENTIRE feederlines have 0 PV generation capacity.")

2306, or 79% of feederlines contain PV generation capacity = 0
1633, or 56 % of ENTIRE feederlines have 0 load capacity.


### Load capacity

In [46]:
# how many observations have load capacity = to 0?
load_0 = pge_circuits[pge_circuits['LoadCapaci'] == 0].shape[0]

print(f"{load_0}, or {round(load_0/nrow * 100)}% of observations contain load capacity = 0")

292675, or 40% of observations contain load capacity = 0


In [62]:
# how many feeder lines?
load_0_feeders = pge_circuits[pge_circuits['LoadCapaci'] == 0].groupby('FeederId').count().shape[0]
df = pge_circuits.groupby('FeederId').sum('LoadCapaci')
load_0_feeders_complete = df[df['LoadCapaci'] == 0].count().iloc[0]

print(f"{load_0_feeders}, or {round(load_0_feeders/nrow_feeders * 100)}% of feederlines contain a segement with load capacity = 0")
print(f"{load_0_feeders_complete}, or {round(load_0_feeders_complete/nrow_feeders * 100)} % of ENTIRE feederlines have 0 load capacity.")

1111, or 38% of feederlines contain a segement with load capacity = 0
453, or 15 % of ENTIRE feederlines have 0 load capacity.


In [56]:
df = pge_circuits.groupby('FeederId').sum('LoadCapaci')
load_0_feeders = df[df['LoadCapaci'] == 0].count()

LoadCapaci    453
voltage_kv    453
phase_cnt     453
GenCapacit    453
GenericPVC    453
GenCapac_1    453
GenericCap    453
Publish       453
SHAPE_Leng    453
dtype: int64

# Calculating Hosting Capacity for a Feederline with Values!

In [67]:
pge_circuits.head()

,FeederId,FeederName,globalid,CSV_LineSe,LoadCapaci,voltage_kv,phase_cnt,limiting_m,limiting_c,ICA_Analys,lica_analy,Division,GenCapacit,GenericPVC,GenCapac_1,GenericCap,limiting_1,limiting_2,limiting_3,limiting_4,limiting_5,limiting_6,limiting_7,limiting_8,ScreenL,Publish,Last_Updat,SHAPE_Leng,geometry
0,062541102,MERIDIAN 1102,{3F991049-BA44-489F-A403-DA79E95B5F6A},3862041,0,12.0,3,None,None,May 2024,May 2024,Sacramento,0,0,0,0,None,None,None,None,None,None,None,None,Unlikely to pass,1,2025-06-02,145.642120,"LINESTRING (-121.95921 39.12370, -121.95951 39..."
1,043302102,MONROE 2102,{65E86C65-2474-4DE9-831A-73C5C6C88469},5458148,2380,21.0,3,None,None,Feb 2025,Feb 2025,Sonoma,60,70,2440,2440,None,None,None,None,None,None,None,None,Likely to pass,1,2025-06-02,9.102052,"LINESTRING (-122.73809 38.48070, -122.73809 38..."
2,063172101,MADISON 2101,{26A0B689-C08B-4AE6-AAC1-341797C5AC4E},3180180,0,21.0,3,None,None,Dec 2024,Dec 2024,Sacramento,0,0,0,0,None,None,None,None,None,None,None,None,Unlikely to pass,1,2025-06-02,4.571886,"LINESTRING (-122.01719 38.69234, -122.01714 38..."
3,162701707,VIERRA 1707,{9B295D61-2151-4336-8039-C935C669DD4D},5317517,420,17.0,3,None,None,May 2025,May 2025,Stockton,0,0,0,0,None,None,None,None,None,None,None,None,Unlikely to pass,1,2025-06-02,47.831056,"LINESTRING (-121.24696 37.75509, -121.24642 37..."
4,254691101,NORCO 1101,{3E99B378-0FDD-4ABD-8D12-6FB6E3E30909},5041662,0,12.0,3,None,None,Apr 2025,Apr 2025,Kern,0,0,0,0,None,None,None,None,None,None,None,None,Likely to pass,1,2025-06-02,11.245946,"LINESTRING (-119.21527 35.29618, -119.21527 35..."


Luckily, one of the feeders we had tried previously – `ALTO 1120` – has values. We will use it in our breakdown!

We'll attempt these calculations using Sofia Rodas' methodology.

In [75]:
pge_circuits[pge_circuits['FeederName'] == "ALTO 1120"].head(2)

,FeederId,FeederName,globalid,CSV_LineSe,LoadCapaci,voltage_kv,phase_cnt,limiting_m,limiting_c,ICA_Analys,lica_analy,Division,GenCapacit,GenericPVC,GenCapac_1,GenericCap,limiting_1,limiting_2,limiting_3,limiting_4,limiting_5,limiting_6,limiting_7,limiting_8,ScreenL,Publish,Last_Updat,SHAPE_Leng,geometry
4709,042031120,ALTO 1120,{52FAEFF8-5B0E-476D-82B6-36403AE73BF7},3097994,985,12.0,3,None,None,Dec 2024,Sep 2025,North Bay,2670,3600,4030,4030,None,None,None,None,None,None,None,None,Likely to pass,1,2025-10-10,21.544083,"LINESTRING (-122.53119 37.89255, -122.53144 37..."
6006,042031120,ALTO 1120,{CDCCED68-E87D-41F7-9089-B848214E02E9},3759242,979,12.0,3,None,None,Dec 2024,Sep 2025,North Bay,270,380,3140,3860,None,None,None,None,None,None,None,None,Likely to pass,1,2025-10-10,96.321812,"LINESTRING (-122.53276 37.89459, -122.53277 37..."


In [78]:
alto = pge_circuits[pge_circuits['FeederName'] == "ALTO 1120"]
alto = alto[['FeederId', 'LoadCapaci', 'GenCapacit', 'GenericPVC', 'GenCapac_1', 'GenericCap', 'geometry']]
alto.head()

,FeederId,LoadCapaci,GenCapacit,GenericPVC,GenCapac_1,GenericCap,geometry
4709,042031120,985,2670,3600,4030,4030,"LINESTRING (-122.53119 37.89255, -122.53144 37..."
6006,042031120,979,270,380,3140,3860,"LINESTRING (-122.53276 37.89459, -122.53277 37..."
6020,042031120,172,290,390,290,390,"LINESTRING (-122.51692 37.88786, -122.51705 37..."
6021,042031120,976,1150,1600,3540,4030,"LINESTRING (-122.51563 37.88752, -122.51571 37..."
6028,042031120,371,860,1080,860,1080,"LINESTRING (-122.50291 37.88471, -122.50288 37..."


## Step 0: Load in the other necessary data

In [ ]:
# create marin bbox (Marin County is where our feederline is)
from shapely.geometry import box
marin_bbox = box(-123.0328, 37.8324, -122.3482, 38.3541)

In [ ]:
# read in building data
buildings = gpd.read_parquet(
    "../../../../../../capstone/electrigrid/data/building_zillow_merges/marin.parquet").set_crs("EPSG:4326")
buildings = gpd.clip(buildings, marin_bbox)

In [ ]:
# read in the census tract data
census_tracts = gpd.read_file("../../../../capstone/electrigrid/data/census/tl_2025_06_tract/tl_2025_06_tract.shp")
census_tracts = gpd.clip(census_tracts, marin_bbox)